# Task 3 – Threshold Rule and Reliability
**Owner:** Shivam Vivek Rai · DSA1 · Action Learning

## 1. Introduction
**Research context.** #6 Wang Z. et al. 2026 (COIN), #5 Abbasi Yadkori et al. 2024 (conformal abstention);
Angelopoulos et al. 2024 (conformal risk control); Geifman & El-Yaniv 2017 (risk-coverage curves);
Doula & Mühlhäuser 2026 (inference-time changes break split-conformal guarantees).

**Objective.** Find the rule for setting the abstention threshold that keeps the error among answered questions
below 5%, 10% and 20%, and what it costs in coverage, in domain and under shift.

**Research question.** Standard conformal guarantees concern coverage, not error among answered questions. Which
rule actually keeps that error below the target?

**Hypothesis.** The naive rule often violates the target; conformal risk control bounds a different quantity;
COIN meets the target most reliably at some cost in coverage.

## 2. Setup

In [ ]:
# --- Setup (implementation guide, section 2). Run once per session. ------------------------------
import os, sys, subprocess, random, json
from pathlib import Path

REPO_URL = "https://github.com/adiarora28/dsa1-conformal-abstention.git"   # <- set once for the team
ON_KAGGLE = Path("/kaggle").exists()
REPO_DIR = Path("/kaggle/working/dsa1-conformal-abstention") if ON_KAGGLE else Path.cwd().resolve().parent
if ON_KAGGLE and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if ON_KAGGLE:  # upgrade only what Kaggle's image may lack; never reinstall torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.56",
                    "accelerate", "bitsandbytes", "sentence-transformers", "xgboost"], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
except ImportError:
    print("PyTorch not installed (this notebook can run on CPU)")

from dsa1 import config as C
from dsa1.cache_io import link_cache_roots
CACHE_ROOT = link_cache_roots("/kaggle/input", "/kaggle/working/cache_merged") if ON_KAGGLE else Path("cache")
print("CACHE_ROOT:", CACHE_ROOT)
if ON_KAGGLE:  # bring outputs of earlier notebooks (attached as inputs) into this working copy
    import shutil
    Path("results").mkdir(exist_ok=True)
    for p in Path("/kaggle/input").rglob("semantic_entropy"):
        if p.is_dir():
            shutil.copytree(p, "results/semantic_entropy", dirs_exist_ok=True)
    for name in ["hand_check.csv", "failure_cases.csv"]:
        hits = sorted(Path("/kaggle/input").rglob(name))
        if hits and not (Path("results") / name).exists():
            shutil.copy(hits[0], Path("results") / name)
            print("copied", hits[0])
for p in sorted(Path(CACHE_ROOT).glob("*/*")):
    print("  ", p.relative_to(CACHE_ROOT))


In [ ]:
# Score tables written by Task 1 (attach Sai Akhil's notebook output as an input on Kaggle).
# Until they exist, token-probability scores are computed directly from the cache so you can start now.
from dsa1 import experiments as E

def find_scores_root():
    for p in Path("/kaggle/input").rglob("scores") if ON_KAGGLE else []:
        if p.is_dir() and any(p.glob("qwen3-*/*.parquet")):
            return p
    return Path("results/scores")

SCORES_ROOT = find_scores_root()
print("SCORES_ROOT:", SCORES_ROOT)

def get_scores(model, dataset):
    path = Path(SCORES_ROOT) / model / f"{dataset}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    print(f"[{model}/{dataset}] no Task 1 score table yet - using token probability from the cache")
    return E.base_scores(CACHE_ROOT, model, dataset)


In [ ]:
from dsa1.splits import load_splits
SPLITS = load_splits("data/splits/splits.json")
SEEDS = [int(s) for s in SPLITS["seeds"]]
TEST_KEYS = {"triviaqa_pool": "test_triviaqa", "bioasq": "test_bioasq", "mmlu_med": "test_mmlu_med"}
print("seeds:", SEEDS, "| sizes (seed 0):", {k: len(v) for k, v in SPLITS["seeds"]["0"].items()})


## 3. Data loading & exploration

In [ ]:
MODELS, ALL = [], {}
for m in C.MODELS:
    try:
        ALL[m] = pd.concat([get_scores(m, ds) for ds in ["triviaqa_pool", "bioasq", "mmlu_med"]]).set_index("qid")
        MODELS.append(m)
    except FileNotFoundError as e:
        print("skip", m, e)
SCORES = [s for s in ["u_tokprob", "u_se", "u_probe"] if any(ALL[m][s].notna().all() for m in MODELS if s in ALL[m])]
POOL = ALL["qwen3-4b"].index[ALL["qwen3-4b"]["dataset"] == "triviaqa_pool"].tolist()
print("models:", MODELS, "| scores:", SCORES, "| TriviaQA pool:", len(POOL))
pd.DataFrame({m: 1 - ALL[m].groupby("dataset")["err"].mean() for m in MODELS}).rename_axis("accuracy")

## 4. Preprocessing
Calibration = 1,000 TriviaQA questions per seed; tests = TriviaQA test, BioASQ, MMLU medical. Uniform weights throughout.

## 5. Model definition – the three rules (see `src/dsa1/rules.py`)
- **V1 naive:** largest t with k_t / n_t ≤ α (n_t ≥ 20).
- **V2 conformal risk control:** largest t with (k_t + 1)/(n + 1) ≤ α; bounds answered-and-wrong over **all** questions.
- **V3 COIN:** largest t, scanned in a fixed order from the most confident, with Clopper-Pearson upper bound on k_t / n_t ≤ α at δ = 0.05.

The unit tests check each rule on a toy problem with a known answer.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_rules_metrics.py"], check=True)

## 6. Running the variations – 5 seeds + 100 random re-splits

In [ ]:
TESTS = {"triviaqa_pool": "test_triviaqa", "bioasq": "test_bioasq", "mmlu_med": "test_mmlu_med"}
rows = []
def run_rules(model, sc, cal_q, test_sets, tag):
    cal = ALL[model].loc[cal_q]
    for ds, tq in test_sets.items():
        tst = ALL[model].loc[tq]
        for r in E.rule_eval(cal[sc].values, cal["err"].values, tst[sc].values, tst["err"].values):
            rows.append({"model": model, "score": sc, "test": ds, **tag, **r})

for model in MODELS:
    for sc in SCORES:
        if ALL[model][sc].isna().any():
            continue
        for s in SEEDS:
            d = SPLITS["seeds"][str(s)]
            run_rules(model, sc, d["cal"], {ds: d[k] for ds, k in TESTS.items()}, {"split": f"seed{s}", "n_cal": 1000})
rng = np.random.default_rng(2026)
BIO = SPLITS["seeds"]["0"]["test_bioasq"]; MED = SPLITS["seeds"]["0"]["test_mmlu_med"]
for b in range(100):
    perm = rng.permutation(POOL)
    for model in MODELS:
        for sc in SCORES:
            if ALL[model][sc].isna().any():
                continue
            run_rules(model, sc, list(perm[:1000]), {"triviaqa_pool": list(perm[1000:2000]), "bioasq": BIO, "mmlu_med": MED},
                      {"split": f"resplit{b}", "n_cal": 1000})
res = pd.DataFrame(rows)
res.to_csv("results/metrics/task3_results.csv", index=False)
print(len(res), "rows")

## 7. Evaluation – violation frequency, realised error, coverage

In [ ]:
from dsa1.metrics import mean_std
main = res[(res["model"] == "qwen3-4b") & (res["score"] == "u_tokprob")]
viol = main.groupby(["test", "rule", "target"]).agg(
    violation_freq=("violated", "mean"), realised=("error_among_answered", "mean"),
    coverage=("coverage", "mean"), answered_and_wrong=("answered_and_wrong_rate", "mean"),
    abstain_all=("n_answered", lambda v: float((v == 0).mean())), n_answered=("n_answered", "mean"))
# sampling noise of the realised error on a finite test set: sqrt(a(1-a)/n_answered)
viol["test_noise_sd"] = np.sqrt(viol.index.get_level_values("target") * (1 - viol.index.get_level_values("target")) / viol["n_answered"].clip(lower=1))
viol.round(3)

In [ ]:
# Compute cost per answered question (GPU seconds), by model and score, 10% target, COIN, in domain
rows = []
for model in MODELS:
    for sc in SCORES:
        c = E.cost_per_question(ALL[model].reset_index(), sc).mean()
        sub = res[(res["model"] == model) & (res["score"] == sc) & (res["rule"] == "coin") & (res["target"] == 0.10)]
        for ds, g in sub.groupby("test"):
            rows.append({"model": model, "score": sc, "test": ds, "gpu_s_per_question": c,
                         "coverage": g["coverage"].mean(), "gpu_s_per_answered": c / max(g["coverage"].mean(), 1e-9)})
cost = pd.DataFrame(rows); cost.to_csv("results/metrics/task3_cost.csv", index=False); cost.round(3)

## 8. Comparison & analysis

In [ ]:
# Key figure – risk-coverage curves with each rule's threshold marked (Qwen3-4B, token probability, seed 0, target 10%)
from dsa1.metrics import risk_coverage_curve, aurc
from dsa1.rules import naive_threshold, crc_threshold, coin_threshold, evaluate_threshold
d = SPLITS["seeds"]["0"]; cal = ALL["qwen3-4b"].loc[d["cal"]]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (ds, key) in zip(axes, TESTS.items()):
    tst = ALL["qwen3-4b"].loc[d[key]]
    cov, risk = risk_coverage_curve(tst["u_tokprob"], tst["err"])
    ax.plot(cov, risk, c="k", lw=1, label=f"risk-coverage (AURC {aurc(tst['u_tokprob'], tst['err']):.3f})")
    for name, fn in [("naive", naive_threshold), ("crc", crc_threshold), ("coin", coin_threshold)]:
        ev = evaluate_threshold(fn(cal["u_tokprob"].values, cal["err"].values, 0.10), tst["u_tokprob"], tst["err"])
        if ev["n_answered"]:
            ax.scatter(ev["coverage"], ev["error_among_answered"], s=80, label=name, zorder=3)
    ax.axhline(0.10, ls="--", c="grey"); ax.set_title(f"Qwen3-4B – {ds}"); ax.set_xlabel("coverage"); ax.set_ylabel("error among answered")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("results/figures/task3_risk_coverage.png", dpi=150); plt.show()

In [ ]:
# Realised error versus target per rule (points = mean over 105 splits, bars = std)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, ds in zip(axes, TESTS):
    g = main[main["test"] == ds].groupby(["rule", "target"])["error_among_answered"].agg(["mean", "std"]).reset_index()
    for i, (r, gr) in enumerate(g.groupby("rule")):
        ax.errorbar(gr["target"] + (i - 1) * 0.004, gr["mean"], yerr=gr["std"], fmt="o", capsize=3, label=r)
    ax.plot([0, 0.25], [0, 0.25], ":", c="grey"); ax.set_title(ds); ax.set_xlabel("target"); ax.legend()
axes[0].set_ylabel("realised error among answered")
plt.tight_layout(); plt.savefig("results/figures/task3_realised_vs_target.png", dpi=150); plt.show()

In [ ]:
# Deeper analysis 1 – model size: coverage at each target vs cost (COIN, token probability, in domain and BioASQ)
g = res[(res["score"] == "u_tokprob") & (res["rule"] == "coin")].groupby(["model", "test", "target"])["coverage"].mean().reset_index()
g = g.merge(cost[cost["score"] == "u_tokprob"][["model", "test", "gpu_s_per_question"]], on=["model", "test"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ds in zip(axes, ["triviaqa_pool", "bioasq"]):
    for tgt, gt in g[g["test"] == ds].groupby("target"):
        ax.plot(gt["gpu_s_per_question"], gt["coverage"], marker="o", label=f"target {tgt:.0%}")
        for _, r in gt.iterrows():
            ax.annotate(r["model"].replace("qwen3-", ""), (r["gpu_s_per_question"], r["coverage"]), fontsize=8)
    ax.set_title(ds); ax.set_xlabel("GPU s / question"); ax.set_ylabel("coverage with COIN"); ax.legend()
plt.tight_layout(); plt.savefig("results/figures/task3_model_size.png", dpi=150); plt.show()

In [ ]:
# Deeper analysis 2 – calibration size: violation frequency per rule for n_cal = 250, 500, 1000 (100 re-splits, Qwen3-4B)
rows = []
rng = np.random.default_rng(7)
for n_cal in [250, 500, 1000]:
    for b in range(100):
        perm = rng.permutation(POOL)
        cal = ALL["qwen3-4b"].loc[list(perm[:n_cal])]; tst = ALL["qwen3-4b"].loc[list(perm[1000:2000])]
        for r in E.rule_eval(cal["u_tokprob"].values, cal["err"].values, tst["u_tokprob"].values, tst["err"].values):
            rows.append({"n_cal": n_cal, **r})
cs = pd.DataFrame(rows)
cs.to_csv("results/metrics/task3_calibration_size.csv", index=False)
p = cs.pivot_table(index="n_cal", columns=["target", "rule"], values="violated", aggfunc="mean")
display(p.round(3))
p.xs(0.10, axis=1, level="target").plot(marker="o"); plt.ylabel("violation frequency (target 10%)")
plt.title("Calibration size vs reliability"); plt.savefig("results/figures/task3_calibration_size.png", dpi=150); plt.show()

In [ ]:
# Deeper analysis 3 – precision shift: thresholds from the float16 cache applied to 8-bit and 4-bit answers
from dsa1.cache_io import cache_dir, load_records
from dsa1.grading import normalize_answer
from dsa1.rules import naive_threshold, crc_threshold, coin_threshold
sub = json.load(open("data/splits/precision_subset.json"))
cal = ALL["qwen3-4b"].loc[SPLITS["seeds"]["0"]["cal"]]
rows = []
for ds in ["triviaqa_pool", "bioasq"]:
    ref_rec = load_records(cache_dir(CACHE_ROOT, "qwen3-4b", ds)).set_index("qid")
    for prec in ["fp16", "8bit", "4bit"]:
        try:
            if prec == "fp16":
                q = ALL["qwen3-4b"].loc[sub[ds]].reset_index(); changed = 0.0
            else:
                q = E.base_scores(CACHE_ROOT, "qwen3-4b", ds, precision=prec, suffix="_subset")
                rec = load_records(cache_dir(CACHE_ROOT, "qwen3-4b", ds, prec).parent / f"{ds}_subset").set_index("qid")
                changed = float(np.mean([normalize_answer(rec.loc[k, "answer"]) != normalize_answer(ref_rec.loc[k, "answer"]) for k in rec.index]))
        except FileNotFoundError:
            print("missing", prec, ds); continue
        for name, fn in [("naive", naive_threshold), ("crc", crc_threshold), ("coin", coin_threshold)]:
            for a in C.TARGETS:
                ev = evaluate_threshold(fn(cal["u_tokprob"].values, cal["err"].values, a), q["u_tokprob"], q["err"])
                rows.append({"dataset": ds, "precision": prec, "rule": name, "target": a, "accuracy": 1 - q["err"].mean(),
                             "answers_changed": changed, **ev})
prec_tab = pd.DataFrame(rows); prec_tab.to_csv("results/metrics/task3_precision_shift.csv", index=False)
prec_tab.pivot_table(index=["dataset", "precision"], columns=["rule", "target"], values="error_among_answered").round(3)

### Confusion matrix and demo (for the defense)

In [ ]:
# Confusion matrix: answered/abstained × right/wrong per rule, target 10%, seed 0, in domain and on BioASQ
from dsa1.rules import naive_threshold, crc_threshold, coin_threshold
d0 = SPLITS["seeds"]["0"]; cal = ALL["qwen3-4b"].loc[d0["cal"]]
RULE_FNS = {"naive": naive_threshold, "crc": crc_threshold, "coin": coin_threshold}
thresholds = {r: fn(cal["u_tokprob"].values, cal["err"].values, 0.10) for r, fn in RULE_FNS.items()}
rows = []
for ds, key in [("triviaqa_pool", "test_triviaqa"), ("bioasq", "test_bioasq")]:
    tst = ALL["qwen3-4b"].loc[d0[key]]
    panels = []
    for r, t in thresholds.items():
        c = E.abstention_confusion(tst["u_tokprob"], tst["err"], t)
        panels.append((r, c)); rows.append({"test": ds, "rule": r, "threshold": t, **c})
    E.plot_confusions(panels, "abstention", f"results/figures/task3_confusion_{ds}.png", f"Qwen3-4B · {ds} · target 10% · seed 0")
    plt.show()
pd.DataFrame(rows).to_csv("results/metrics/task3_confusion.csv", index=False)
pd.DataFrame(rows)

In [ ]:
# Demo – inference on test samples: five BioASQ questions, the model's answer, and each rule's decision at 10%
from dsa1.cache_io import cache_dir, load_records
pd.set_option("display.max_colwidth", 70)
rec = load_records(cache_dir(CACHE_ROOT, "qwen3-4b", "bioasq")).set_index("qid")
tst = ALL["qwen3-4b"].loc[d0["test_bioasq"]]
pick = tst.sample(5, random_state=3)
demo = pd.DataFrame({
    "question": [rec.loc[q, "question"] for q in pick.index],
    "model answer": [rec.loc[q, "answer"] for q in pick.index],
    "correct": (pick["err"] == 0).values,
    "uncertainty": pick["u_tokprob"].round(3).values,
    **{r: np.where(pick["u_tokprob"] <= t, "answer", "route to staff") for r, t in thresholds.items()},
})
demo.to_csv("results/metrics/task3_demo_examples.csv", index=False)
demo

## 9. Discussion
- How often does each rule violate its target, in domain and under shift? Is COIN's violation rate near δ in domain?
- What does conformal risk control guarantee here, and why does it differ from error among answered?
- Coverage cost of COIN's guarantee; infeasible targets on BioASQ.
- Model size and calibration size: what is worth paying for?
- Precision shift: do 8-bit and 4-bit answers keep the float16 threshold's promise?
- MVP: which rule routes questions to staff, at which target.

## 10. References

- Bibliography numbers refer to `DSA 1 - Bibliography.xlsx`.
- Datasets: TriviaQA (Joshi et al., 2017), BioASQ Task B 14b (Krithara et al., 2023), MMLU (Hendrycks et al., 2021).
- Models: Qwen3 (Qwen Team, 2025); DeBERTa-large-MNLI (He et al., 2021); all-MiniLM-L6-v2 (Reimers & Gurevych, 2019).
- Code: shared `dsa1` package in this repository; LM-Polygraph (Vashurin et al., 2025) used for cross-checks;
  DS-CP reference code https://github.com/zhexiaolin/CP (Lin et al., 2026).
